# 🚀 NUWA-MLTB — Heroku Deployment

| | |
|---|---|
| **Repo** | [shubhamdubey18/NUWA-MLTB](https://github.com/shubhamdubey18/NUWA-MLTB) |
| **Stack** | Docker (heroku.yml) |
| **Runtime** | `5hojib/aeon` base image |

> **Steps:** Run each cell top-to-bottom. Fill in your values before running.
>
> ⚠️ Keep your tokens private — do not share this notebook after filling credentials.

## Step 1 — Install Heroku CLI & Authenticate

In [ ]:
# @title 🔑 Heroku Login
# @markdown Get your API key from: https://dashboard.heroku.com/account

HEROKU_EMAIL = ""  # @param {type:"string"}
HEROKU_API_KEY = ""  # @param {type:"string"}

# ── install Heroku CLI ─────────────────────────────────────────────────────────
import subprocess, os, textwrap

def run(cmd, **kw):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.stdout: print(result.stdout.strip())
    if result.stderr: print(result.stderr.strip())
    return result

print("Installing Heroku CLI...")
run("curl -sL https://cli-assets.heroku.com/install.sh | sh")

# ── write .netrc for passwordless auth ────────────────────────────────────────
netrc = textwrap.dedent(f"""
    machine api.heroku.com
      login {HEROKU_EMAIL}
      password {HEROKU_API_KEY}
    machine git.heroku.com
      login {HEROKU_EMAIL}
      password {HEROKU_API_KEY}
""").strip()

with open(os.path.expanduser("~/.netrc"), "w") as f:
    f.write(netrc)
os.chmod(os.path.expanduser("~/.netrc"), 0o600)

os.environ["HEROKU_API_KEY"] = HEROKU_API_KEY

result = run("heroku auth:whoami")
if HEROKU_EMAIL in result.stdout:
    print(f"\n✅ Logged in as: {HEROKU_EMAIL}")
else:
    print("\n❌ Login failed — check your email and API key.")

## Step 2 — Create Heroku App & Clone Repo

In [ ]:
# @title 🏗️ App Setup
# @markdown Leave APP_NAME blank to let Heroku auto-generate a name.

APP_NAME = ""   # @param {type:"string"}
REGION   = "us" # @param ["us", "eu"]

import os, subprocess

def run(cmd, **kw):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.stdout: print(result.stdout.strip())
    if result.stderr: print(result.stderr.strip())
    return result

# ── create app ─────────────────────────────────────────────────────────────────
name_flag = f"--app {APP_NAME}" if APP_NAME.strip() else ""
result = run(f"heroku create {APP_NAME.strip()} --region {REGION} --stack container")

# detect actual app name (in case auto-generated)
import re
match = re.search(r'https://([\w-]+)\.herokuapp\.com', result.stdout + result.stderr)
if match:
    APP_NAME = match.group(1)

print(f"\n📦 App name: {APP_NAME}")
os.environ["APP_NAME"] = APP_NAME

# ── clone repo ─────────────────────────────────────────────────────────────────
if not os.path.isdir("/content/NUWA-MLTB"):
    print("\nCloning NUWA-MLTB...")
    run("git clone https://github.com/shubhamdubey18/NUWA-MLTB /content/NUWA-MLTB")
else:
    print("\nRepo already cloned.")

run(f"heroku git:remote --app {APP_NAME}", cwd="/content/NUWA-MLTB")
print("✅ Heroku remote set.")

## Step 3 — Required Configuration

In [ ]:
# @title ⚙️ Required Config Vars
# @markdown Get Telegram API credentials from https://my.telegram.org

BOT_TOKEN      = ""  # @param {type:"string"}
OWNER_ID       = ""  # @param {type:"string"}
TELEGRAM_API   = ""  # @param {type:"string"}
TELEGRAM_HASH  = ""  # @param {type:"string"}
DATABASE_URL   = ""  # @param {type:"string"} MongoDB URI — get free at mongodb.com/atlas
BASE_URL       = ""  # @param {type:"string"} e.g. https://your-app-name.herokuapp.com
BASE_URL_PORT  = "80" # @param {type:"string"}

required_vars = {
    "BOT_TOKEN":     BOT_TOKEN,
    "OWNER_ID":      OWNER_ID,
    "TELEGRAM_API":  TELEGRAM_API,
    "TELEGRAM_HASH": TELEGRAM_HASH,
    "DATABASE_URL":  DATABASE_URL,
    "BASE_URL":      BASE_URL,
    "BASE_URL_PORT": BASE_URL_PORT,
}

missing = [k for k, v in required_vars.items() if not str(v).strip()]
if missing:
    print(f"⚠️  Fill in required fields: {', '.join(missing)}")
else:
    print("✅ All required fields provided.")

## Step 4 — Optional Configuration

In [ ]:
# @title ⚙️ Optional Config Vars
# @markdown Leave blank to use defaults.

UPSTREAM_REPO       = "https://github.com/shubhamdubey18/NUWA-MLTB" # @param {type:"string"}
UPSTREAM_BRANCH     = "master"  # @param {type:"string"}
DEFAULT_UPLOAD      = "rc"      # @param ["rc", "gd"]
AUTHORIZED_CHATS    = ""        # @param {type:"string"}
SUDO_USERS          = ""        # @param {type:"string"}
USER_SESSION_STRING = ""        # @param {type:"string"}
CMD_SUFFIX          = ""        # @param {type:"string"}
GDRIVE_ID           = ""        # @param {type:"string"}
INDEX_URL           = ""        # @param {type:"string"}
RCLONE_PATH         = ""        # @param {type:"string"}
LEECH_SPLIT_SIZE    = "0"       # @param {type:"string"}
LEECH_DUMP_CHAT     = ""        # @param {type:"string"}
STATUS_LIMIT        = "4"       # @param {type:"string"}
FILELION_API        = ""        # @param {type:"string"}
STREAMWISH_API      = ""        # @param {type:"string"}
QUEUE_ALL           = "0"       # @param {type:"string"}
QUEUE_DOWNLOAD      = "0"       # @param {type:"string"}
QUEUE_UPLOAD        = "0"       # @param {type:"string"}
RSS_DELAY           = "600"     # @param {type:"string"}
RSS_CHAT            = ""        # @param {type:"string"}
JD_EMAIL            = ""        # @param {type:"string"}
JD_PASS             = ""        # @param {type:"string"}

optional_vars = {
    "UPSTREAM_REPO":       UPSTREAM_REPO,
    "UPSTREAM_BRANCH":     UPSTREAM_BRANCH,
    "DEFAULT_UPLOAD":      DEFAULT_UPLOAD,
    "AUTHORIZED_CHATS":    AUTHORIZED_CHATS,
    "SUDO_USERS":          SUDO_USERS,
    "USER_SESSION_STRING": USER_SESSION_STRING,
    "CMD_SUFFIX":          CMD_SUFFIX,
    "GDRIVE_ID":           GDRIVE_ID,
    "INDEX_URL":           INDEX_URL,
    "RCLONE_PATH":         RCLONE_PATH,
    "LEECH_SPLIT_SIZE":    LEECH_SPLIT_SIZE,
    "LEECH_DUMP_CHAT":     LEECH_DUMP_CHAT,
    "STATUS_LIMIT":        STATUS_LIMIT,
    "FILELION_API":        FILELION_API,
    "STREAMWISH_API":      STREAMWISH_API,
    "QUEUE_ALL":           QUEUE_ALL,
    "QUEUE_DOWNLOAD":      QUEUE_DOWNLOAD,
    "QUEUE_UPLOAD":        QUEUE_UPLOAD,
    "RSS_DELAY":           RSS_DELAY,
    "RSS_CHAT":            RSS_CHAT,
    "JD_EMAIL":            JD_EMAIL,
    "JD_PASS":             JD_PASS,
}

print("✅ Optional vars staged.")

## Step 5 — Push Config & Deploy

In [ ]:
# @title 🚀 Set Config Vars & Deploy to Heroku

import subprocess, os, shlex

APP = os.environ.get("APP_NAME", "") or globals().get("APP_NAME", "")
if not APP:
    APP = input("Enter your Heroku app name: ").strip()

def run(cmd, cwd=None):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if result.stdout: print(result.stdout.strip())
    if result.stderr: print(result.stderr.strip())
    return result

# ── safe fallback if Steps 3/4 were skipped ───────────────────────────────────
_req = globals().get("required_vars", {})
_opt = globals().get("optional_vars", {})
all_vars = {**_req, **_opt}

if not all_vars:
    print("⚠️  No config vars found — please run Steps 3 and 4 first, then re-run this cell.")
else:
    config_str = " ".join(
        f"{k}={shlex.quote(str(v))}"
        for k, v in all_vars.items()
        if str(v).strip()
    )

    print("Setting config vars...")
    run(f"heroku config:set {config_str} --app {APP}")

    # ── git push to heroku ─────────────────────────────────────────────────────
    print("\nDeploying to Heroku (Docker build — this may take 5-10 min)...")
    result = run("git push heroku master --force", cwd="/content/NUWA-MLTB")

    if result.returncode == 0:
        print(f"\n✅ Deployment complete!")
        print(f"🔗 App URL : https://{APP}.herokuapp.com")
        print(f"📊 Dashboard: https://dashboard.heroku.com/apps/{APP}")
    else:
        print("\n❌ Deployment failed — check logs in Step 6.")

## Step 6 — Utilities

In [ ]:
# @title 📋 View Logs

import subprocess, os
APP = os.environ.get("APP_NAME", "")
if not APP:
    APP = input("Enter your app name: ").strip()

LOG_LINES = 100  # @param {type:"integer"}

result = subprocess.run(
    f"heroku logs --tail=false --num {LOG_LINES} --app {APP}",
    shell=True, capture_output=True, text=True
)
print(result.stdout or result.stderr)

In [ ]:
# @title 🔄 Restart Dyno

import subprocess, os
APP = os.environ.get("APP_NAME", "")
if not APP:
    APP = input("Enter your app name: ").strip()

result = subprocess.run(f"heroku restart --app {APP}", shell=True, capture_output=True, text=True)
print(result.stdout or result.stderr)
print(f"✅ Dyno restarted for {APP}")

In [ ]:
# @title ⚙️ Update Config Var (single)

import subprocess, os, shlex
APP = os.environ.get("APP_NAME", "")
if not APP:
    APP = input("Enter your app name: ").strip()

VAR_NAME  = ""  # @param {type:"string"}
VAR_VALUE = ""  # @param {type:"string"}

if VAR_NAME.strip():
    result = subprocess.run(
        f"heroku config:set {VAR_NAME}={shlex.quote(VAR_VALUE)} --app {APP}",
        shell=True, capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
    print(f"✅ {VAR_NAME} updated.")
else:
    print("⚠️  Enter a variable name.")

In [ ]:
# @title 🗑️ Destroy App (IRREVERSIBLE)
# @markdown Type your app name exactly to confirm destruction.

import subprocess
CONFIRM_APP_NAME = ""  # @param {type:"string"}

if CONFIRM_APP_NAME.strip():
    result = subprocess.run(
        f"heroku apps:destroy --app {CONFIRM_APP_NAME} --confirm {CONFIRM_APP_NAME}",
        shell=True, capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
else:
    print("⚠️  Enter the app name to confirm destruction.")